# 01 — Ingesta y extracción de documentos PDF

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Prototipado del pipeline de ingesta

---

Este notebook cubre la primera fase del pipeline RAG: la carga, extracción
y estructuración del contenido de un documento PDF científico de química
mediante **Docling** (IBM).

El output de este notebook es la entrada del notebook `02_embeddings.ipynb`.

In [1]:
"""
Notebook: 01_ingesta.ipynb

Objetivo:
    Cargar un documento PDF científico de química y extraer su contenido
    estructurado (texto, tablas y elementos especiales) usando Docling (IBM),
    tal y como se haría en la fase de ingesta de un pipeline RAG en producción.

    Este notebook cubre la carga, extracción con Docling, limpieza
    universal y específica por documento, auditoría del resultado
    y persistencia del documento procesado. El chunking y la
    vectorización se abordan en 02_embeddings.ipynb.

Fuente de datos:
    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,
    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical
    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)
    complexes, with apoptosis-inducing properties in cisplatin-resistant
    neuroblastoma cells. Frontiers in Chemistry. 2024.
    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/

    Documento utilizado exclusivamente con fines de investigación y desarrollo.
    No se distribuye ni se incluye en el repositorio.

Autor:   Jesús Rodríguez
Fecha:   2026-04-30
Versión: 1.1.0
"""

'\nNotebook: 01_ingesta.ipynb\n\nObjetivo:\n    Cargar un documento PDF científico de química y extraer su contenido\n    estructurado (texto, tablas y elementos especiales) usando Docling (IBM),\n    tal y como se haría en la fase de ingesta de un pipeline RAG en producción.\n\n    Este notebook cubre la carga, extracción con Docling, limpieza\n    universal y específica por documento, auditoría del resultado\n    y persistencia del documento procesado. El chunking y la\n    vectorización se abordan en 02_embeddings.ipynb.\n\nFuente de datos:\n    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,\n    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical\n    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)\n    complexes, with apoptosis-inducing properties in cisplatin-resistant\n    neuroblastoma cells. Frontiers in Chemistry. 2024.\n    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/\n\n    Documento utilizado exclusiva

## 1. Configuración del entorno

In [2]:
# Detección del entorno de ejecución (Colab vs local)
try:
    from google.colab import drive
    IN_COLAB = True
    print("Entorno detectado: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Entorno detectado: local")

Entorno detectado: Google Colab


In [3]:
# Montaje de Google Drive (solo en Colab)
if IN_COLAB:
    drive.mount('/content/drive')
    print("Google Drive montado correctamente")

Mounted at /content/drive
Google Drive montado correctamente


In [4]:
# Verificación del entorno de ejecución
import sys
import platform

print(f"Python : {sys.version}")
print(f"Sistema: {platform.system()} {platform.release()}")

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sistema: Linux 6.6.122+


## 2. Instalación de dependencias

In [5]:
# Instalación de Docling (IBM) para extracción estructurada de PDFs científicos
# Se instala en modo silencioso (-q) para reducir el output en Colab
%pip install docling -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 12.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.4/519.4 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 70.3 MB/s eta 0:00:00
   ━━

In [6]:
# Verificación de la instalación de Docling
from docling.document_converter import DocumentConverter

print("Docling importado correctamente")

Docling importado correctamente


## 3. Definición de rutas

In [7]:
# Librería estándar para manejo de rutas
from pathlib import Path

# Ruta raíz del proyecto en Google Drive
PROYECTO_RAIZ = Path('/content/drive/MyDrive/chem-rag-assistant')

# Ruta al directorio de documentos de entrada
DIR_RAW = PROYECTO_RAIZ / 'data' / 'raw'

# Ruta al directorio de salida para artefactos del notebook
DIR_OUTPUT = PROYECTO_RAIZ / 'data' / 'processed'
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

# Nombre del documento de prueba
NOMBRE_PDF = 'PMC10967698.pdf'
RUTA_PDF = DIR_RAW / NOMBRE_PDF

# Validación de existencia del archivo antes de continuar
assert RUTA_PDF.exists(), (
    f"Archivo no encontrado: {RUTA_PDF}\n"
    f"Asegúrate de que el PDF está en: {DIR_RAW}"
)

print(f"Documento localizado : {RUTA_PDF}")
print(f"Tamaño del archivo   : {RUTA_PDF.stat().st_size / 1024:.1f} KB")

Documento localizado : /content/drive/MyDrive/chem-rag-assistant/data/raw/PMC10967698.pdf
Tamaño del archivo   : 785.0 KB


## 4. Extracción del documento con Docling

In [8]:
# Inicialización del convertidor de Docling
# DocumentConverter detecta automáticamente el tipo de documento
# y aplica el pipeline de extracción adecuado
converter = DocumentConverter()

print("Iniciando extracción del documento...")
resultado = converter.convert(str(RUTA_PDF))
print("Extracción completada")

Iniciando extracción del documento...


[INFO] 2026-05-19 13:24:52,165 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-19 13:24:52,188 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-19 13:24:52,211 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-19 13:24:53,341 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-05-19 13:24:54,111 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-19 13:24:54,121 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-19 13:24:55,312 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-19 13:24:55,315 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-19 13:24:55,323 [RapidOCR] download_file.py:68: Initiating download: https

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Extracción completada


In [9]:
# Exportación del documento extraído a formato Markdown
# Markdown preserva la estructura del documento (títulos, tablas, listas)
# y es el formato óptimo como entrada para el pipeline de chunking
documento_md = resultado.document.export_to_markdown()

print(f"Caracteres extraídos : {len(documento_md):,}")
print(f"Palabras aproximadas : {len(documento_md.split()):,}")

Caracteres extraídos : 59,614
Palabras aproximadas : 10,851


## 5. Limpieza del texto extraído

In [10]:
# Fase 5a: Limpieza universal
# Elimina artefactos comunes a cualquier PDF científico extraído
# con Docling. Estas reglas son independientes del documento
# y forman la base del pipeline reutilizable.
import re

# Eliminar placeholders de imágenes generados por Docling
documento_limpio = re.sub(r'<!--\s*image\s*-->', '', documento_md)

# Colapsar múltiples saltos de línea consecutivos en uno solo
documento_limpio = re.sub(r'\n{3,}', '\n\n', documento_limpio)

# Eliminar caracteres Unicode problemáticos generados por OCR
# (ligaduras tipográficas no reconocidas como fl, fi, ff)
documento_limpio = re.sub(r'[\ue000-\uf8ff]', '', documento_limpio)

# Eliminar espacios en blanco al inicio y al final del documento
documento_limpio = documento_limpio.strip()

print("Limpieza universal completada")
print(f"Caracteres antes de limpieza : {len(documento_md):,}")
print(f"Caracteres tras limpieza     : {len(documento_limpio):,}")
print(
    f"Reducción                    : "
    f"{len(documento_md) - len(documento_limpio):,} caracteres"
)

Limpieza universal completada
Caracteres antes de limpieza : 59,614
Caracteres tras limpieza     : 59,256
Reducción                    : 358 caracteres


In [11]:
# Fase 5b: Limpieza específica por documento
# Aplica reglas opcionales definidas en config/cleaning_rules.yaml
# Si no existe configuración para el documento actual, se omite
# esta fase sin afectar al pipeline.
import yaml
from pathlib import Path

RUTA_CONFIG = PROYECTO_RAIZ / 'config' / 'cleaning_rules.yaml'
NOMBRE_DOC  = RUTA_PDF.stem  # 'PMC10967698'

if RUTA_CONFIG.exists():
    with open(RUTA_CONFIG, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)

    reglas = config.get(NOMBRE_DOC, {})

    if reglas:
        # Eliminar patrones específicos del documento
        for patron in reglas.get('remove_patterns', []):
            documento_limpio = re.sub(
                patron, '', documento_limpio, flags=re.IGNORECASE
            )

        # Eliminar bloques de afiliaciones institucionales
        if reglas.get('remove_affiliations'):
            documento_limpio = re.sub(
                r'\n[a-z]\s+[A-Z][^\n]{20,}\n',
                '\n',
                documento_limpio
            )

        # Corregir errores OCR básicos conocidos en papers científicos
        if reglas.get('fix_ocr_errors'):
            ocr_fixes = {
                r' rst '      : ' first ',
                r'O  en '     : 'Often ',
                r'signi  cant': 'significant',
                r'di  erent'  : 'different',
                r'a  ect'     : 'affect',
                r'e  ect'     : 'effect',
            }
            for patron, reemplazo in ocr_fixes.items():
                documento_limpio = re.sub(
                    patron, reemplazo, documento_limpio, flags=re.IGNORECASE
                )

        # Eliminar sección de referencias al final del documento
        if reglas.get('remove_references_section'):
            documento_limpio = re.sub(
                r'## Notes and references.*$',
                '',
                documento_limpio,
                flags=re.DOTALL | re.IGNORECASE
            )

        # Eliminar encabezados Markdown vacíos o residuales
        # generados tras la eliminación de patrones específicos
        documento_limpio = re.sub(
            r'^##\s*$', '', documento_limpio, flags=re.MULTILINE
        )

        # Colapsar saltos de línea generados por las eliminaciones
        documento_limpio = re.sub(r'\n{3,}', '\n\n', documento_limpio)

        print(f"Limpieza específica aplicada para: {NOMBRE_DOC}")
        print(
            f"  Caracteres finales: {len(documento_limpio):,}"
        )
    else:
        print(f"Sin reglas específicas para: {NOMBRE_DOC}")
else:
    print("Archivo de configuración no encontrado — omitiendo limpieza específica")

Limpieza específica aplicada para: PMC10967698
  Caracteres finales: 49,956


## 6. Auditoría del resultado de limpieza

In [13]:
# Verificación visual del texto limpio mediante un chunking
# exploratorio. El objetivo no es generar los chunks definitivos
# sino detectar si quedan artefactos problemáticos antes de
# persistir el documento procesado.
# Esta celda no modifica ningún dato — es solo diagnóstico.
%pip install langchain-text-splitters -q

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter_auditoria = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

chunks_auditoria = splitter_auditoria.split_text(documento_limpio)

print(f"Total chunks         : {len(chunks_auditoria)}")
print(f"Chunk más corto      : {min(len(c) for c in chunks_auditoria)} caracteres")
print()
print("PREVIEW — CHUNK 0:")
print("=" * 60)
print(chunks_auditoria[0])
print("=" * 60)

chunks_cortos = [(i, c) for i, c in enumerate(chunks_auditoria) if len(c) < 100]
print(f"\nChunks con menos de 100 caracteres: {len(chunks_cortos)}")
print("-" * 45)
for idx, chunk in chunks_cortos:
    print(f"  Chunk {idx:02d} ({len(chunk)} chars): {repr(chunk)}")

Total chunks         : 76
Chunk más corto      : 15 caracteres

PREVIEW — CHUNK 0:
## Introduction

N-heterocyclic carbenes (NHCs),  first described in 1991, 1 have found many applications. 2 There are several structural features that allow the tuning of their electronic properties. Ring size, the adjacent heteroatoms, N -substituents, and the backbone can be modi  ed. Changing one or more structural properties of a NHC ligand can lead to significantly di ff erent reactivities and stabilities of the resulting complexes. 3 Often several NHC units are combined in multidentate ligands, making use of the chelating e ff ect, and a plethora of multidentate NHC metal complexes has been reported. 4,5

† Electronic supplementary information (ESI) available: Synthetic details, biological studies, analytical data and crystallographic data. CCDC 2299372 -2299374. For ESI and crystallographic data in CIF or other electronic format see 

‡ These authors contributed equally to this work.

Chunks con 

## 7. Validación del contenido extraído

In [14]:
# Inspección de los primeros 2000 caracteres del documento limpio
# para verificar que la limpieza preservó la estructura correctamente
CARACTERES_PREVIEW = 2000

print("=" * 60)
print("PREVIEW DEL DOCUMENTO LIMPIO")
print("=" * 60)
print(documento_limpio[:CARACTERES_PREVIEW])
print("...")
print("=" * 60)

PREVIEW DEL DOCUMENTO LIMPIO

## Introduction

N-heterocyclic carbenes (NHCs),  first described in 1991, 1 have found many applications. 2 There are several structural features that allow the tuning of their electronic properties. Ring size, the adjacent heteroatoms, N -substituents, and the backbone can be modi  ed. Changing one or more structural properties of a NHC ligand can lead to significantly di ff erent reactivities and stabilities of the resulting complexes. 3 Often several NHC units are combined in multidentate ligands, making use of the chelating e ff ect, and a plethora of multidentate NHC metal complexes has been reported. 4,5

† Electronic supplementary information (ESI) available: Synthetic details, biological studies, analytical data and crystallographic data. CCDC 2299372 -2299374. For ESI and crystallographic data in CIF or other electronic format see 

‡ These authors contributed equally to this work.

Synthesis, characterization, and biomedical evaluation of ethyle

In [15]:
# Validación de presencia de términos clave del paper
# Verificamos que Docling extrajo correctamente la terminología
# organometálica (compuestos, metales, técnicas analíticas)
TERMINOS_CLAVE = [
    'Pd',            # Paladio
    'Pt',            # Platino
    'Au',            # Oro
    'NHC',           # N-Heterocyclic Carbene
    'cisplatin',     # Referencia farmacológica
    'apoptosis',     # Mecanismo biológico
    'neuroblastoma', # Línea celular estudiada
]

print("Validación de términos clave en el documento extraído:")
print("-" * 45)
for termino in TERMINOS_CLAVE:
    encontrado = termino.lower() in documento_limpio.lower()
    estado = "OK" if encontrado else "NO ENCONTRADO"
    print(f"  [{estado}]  {termino}")
print("-" * 45)

Validación de términos clave en el documento extraído:
---------------------------------------------
  [OK]  Pd
  [OK]  Pt
  [OK]  Au
  [OK]  NHC
  [OK]  cisplatin
  [OK]  apoptosis
  [OK]  neuroblastoma
---------------------------------------------


## 8. Persistencia del documento procesado

In [16]:
# Guardado del documento limpio en formato Markdown
# Este archivo contiene el texto extraído y procesado por las Secciones 5a y 5b
RUTA_OUTPUT_MD = DIR_OUTPUT / 'PMC10967698_extracted.md'

with open(RUTA_OUTPUT_MD, 'w', encoding='utf-8') as f:
    f.write(documento_limpio)

print(f"Documento guardado en : {RUTA_OUTPUT_MD}")
print(f"Tamaño del archivo    : {RUTA_OUTPUT_MD.stat().st_size / 1024:.1f} KB")

Documento guardado en : /content/drive/MyDrive/chem-rag-assistant/data/processed/PMC10967698_extracted.md
Tamaño del archivo    : 48.9 KB


## 9. Resumen de la ejecución

In [17]:
# Resumen final del proceso de ingesta y limpieza
print("=" * 60)
print("RESUMEN — INGESTA Y LIMPIEZA COMPLETADAS")
print("=" * 60)
print(f"  Documento origen      : {NOMBRE_PDF}")
print(f"  Caracteres originales : {len(documento_md):,}")
print(f"  Caracteres tras limpieza: {len(documento_limpio):,}")
print(f"  Palabras aprox.       : {len(documento_limpio.split()):,}")
print(f"  Output generado       : {RUTA_OUTPUT_MD.name}")
print("=" * 60)
print("Siguiente paso: 02_embeddings.ipynb")

RESUMEN — INGESTA Y LIMPIEZA COMPLETADAS
  Documento origen      : PMC10967698.pdf
  Caracteres originales : 59,614
  Caracteres tras limpieza: 49,956
  Palabras aprox.       : 8,920
  Output generado       : PMC10967698_extracted.md
Siguiente paso: 02_embeddings.ipynb
